In [2]:
import os
import subprocess
from tqdm import tqdm

def convert_mp4_to_mp3(folder_path):
    if not os.path.isdir(folder_path):
        print("La cartella non esiste.")
        return

    # Trova tutti gli mp4
    mp4_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(".mp4"):
                mp4_files.append(os.path.join(root, file))

    if not mp4_files:
        print("Nessun file mp4 trovato.")
        return

    print(f"Trovati {len(mp4_files)} file da convertire.")

    for mp4_path in tqdm(mp4_files, desc="Conversione", unit="file"):
        mp3_path = os.path.splitext(mp4_path)[0] + ".mp3"

        try:
            subprocess.run(
                [
                    "ffmpeg",
                    "-y",
                    "-i", mp4_path,
                    "-vn",
                    "-ab", "320k",
                    mp3_path
                ],
                check=True
            )

            os.remove(mp4_path)

        except subprocess.CalledProcessError:
            print(f"\nErrore nella conversione di: {mp4_path}")

convert_mp4_to_mp3("/Users/sem/Downloads/VideoDownloader")


Trovati 3 file da convertire.


Conversione:   0%|          | 0/3 [00:00<?, ?file/s]ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --cc=clang --host-cflags= --host-ldflags= --enable-gpl --enable-libaom --enable-libdav1d --enable-libharfbuzz --enable-libmp3lame --enable-libopus --enable-libsnappy --enable-libsvtav1 --enable-libtheora --enable-libvorbis --enable-libvpx --enable-libx264 --enable-libx265 --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-demuxer=dash --enable-neon --enable-opencl --enable-audiotoolbox --enable-videotoolbox --disable-htmlpages
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  

In [1]:
import os
import subprocess
import random
from tqdm import tqdm

def get_duration(mp4_path):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            mp4_path
        ],
        capture_output=True, text=True
    )
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None

def convert_mp4_to_mp3(folder_path):
    if not os.path.isdir(folder_path):
        print("La cartella non esiste.")
        return

    mp4_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(".mp4"):
                mp4_files.append(os.path.join(root, file))

    if not mp4_files:
        print("Nessun file mp4 trovato.")
        return

    print(f"Trovati {len(mp4_files)} file da convertire.")

    for mp4_path in tqdm(mp4_files, desc="Conversione", unit="file"):
        mp3_path = os.path.splitext(mp4_path)[0] + ".mp3"
        cover_path = os.path.splitext(mp4_path)[0] + "_cover.jpg"

        try:
            duration = get_duration(mp4_path)
            timestamp = random.uniform(0, duration) if duration else 0

            subprocess.run(
                [
                    "ffmpeg", "-y",
                    "-ss", str(timestamp),
                    "-i", mp4_path,
                    "-frames:v", "1",
                    "-q:v", "2",
                    cover_path
                ],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

            subprocess.run(
                [
                    "ffmpeg", "-y",
                    "-i", mp4_path,
                    "-i", cover_path,
                    "-map", "0:a",
                    "-map", "1:0",
                    "-c:a", "libmp3lame",
                    "-ab", "320k",
                    "-id3v2_version", "3",
                    "-metadata:s:v", "title=Album cover",
                    "-metadata:s:v", "comment=Cover (front)",
                    mp3_path
                ],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

            os.remove(mp4_path)

        except subprocess.CalledProcessError:
            print(f"\nErrore nella conversione di: {mp4_path}")

        finally:
            if os.path.exists(cover_path):
                os.remove(cover_path)

convert_mp4_to_mp3("/Users/sem/Downloads/VideoDownloader")

Trovati 1 file da convertire.


Conversione: 100%|██████████| 1/1 [00:00<00:00,  2.64file/s]


Errore nella conversione di: /Users/sem/Downloads/VideoDownloader/stream_8e56980e.mp4
